In [ ]:
from flask import Flask, request, jsonify
import requests

app_quote = Flask("QuotationService")

DISTRIBUTORS = [
    "http://127.0.0.1:5001/quote",
    "http://127.0.0.1:5002/quote",
    "http://127.0.0.1:5003/quote"
]

@app_quote.route('/get_best_quotes', methods=['POST'])
def get_best_quotes():
    product_ids = request.json['product_ids']
    all_quotes = {}

    # Ask distributors
    for url in DISTRIBUTORS:
        try:
            res = requests.post(url, json={"product_ids": product_ids})
            for pid, info in res.json().items():
                if pid not in all_quotes: all_quotes[pid] = []
                all_quotes[pid].append(info)
        except:
            print(f"⚠️ Warning: Could not reach {url}")

    # Find best price
    best_options = {}
    for pid in product_ids:
        if pid in all_quotes and all_quotes[pid]:
            best_options[pid] = sorted(all_quotes[pid], key=lambda x: x['price'])[0]
        else:
            best_options[pid] = {"error": "Out of stock"}

    return jsonify(best_options)

print("✅ Quotation Service Running on Port 5004...")
app_quote.run(port=5004, use_reloader=False) 
# Note: The cell will stay busy [*]. This is normal!

✅ Quotation Service Running on Port 5004...
 * Serving Flask app 'QuotationService'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /get_best_quotes HTTP/1.1" 200 -
